In [5]:
import pandas as pd
import duckdb
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')

# I\. Load train set

In [7]:
df_train = pd.read_parquet(PROCESSED_DIR / 'omni_train_cleaned.parquet')
df_train.head(5)

,datetime,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc,day_cos,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10,data_split
0,2000-01-01 00:00:00,2000,1,0,0,6.820,-5.940,0.27,-0.08,664.7,3.12,0.999852,0.017213,1.0,0.0,1.000000,0.000000,53,train
1,2000-01-01 00:01:00,2000,1,0,1,6.990,-5.880,1.95,1.08,664.7,3.12,0.999852,0.017213,1.0,0.0,0.994522,0.104528,53,train
2,2000-01-01 00:02:00,2000,1,0,2,6.990,-5.710,2.74,2.24,663.2,3.24,0.999852,0.017213,1.0,0.0,0.978148,0.207912,53,train
3,2000-01-01 00:03:00,2000,1,0,3,6.830,-5.330,3.18,2.78,662.2,3.11,0.999852,0.017213,1.0,0.0,0.951057,0.309017,53,train
4,2000-01-01 00:04:00,2000,1,0,4,6.905,-4.565,2.99,3.76,675.3,2.83,0.999852,0.017213,1.0,0.0,0.913545,0.406737,53,train


In [9]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12135459 entries, 0 to 12358078
Data columns (total 19 columns):
 #   Column               Dtype         
---  ------               -----         
 0   datetime             datetime64[us]
 1   year                 int64         
 2   day                  int64         
 3   hour                 int64         
 4   minute               int64         
 5   mag_avg_nt           float64       
 6   bx_gsm_nt            float64       
 7   by_gsm_nt            float64       
 8   bz_gsm_nt            float64       
 9   flow_speed_km_s      float64       
 10  proton_density_n_cc  float64       
 11  day_cos              float64       
 12  day_sin              float64       
 13  hour_cos             float64       
 14  hour_sin             float64       
 15  minute_cos           float64       
 16  minute_sin           float64       
 17  kp_10                int64         
 18  data_split           object        
dtypes: datetime64[us](1), fl

In [11]:
# Confirm no nulls survived cleaning
print(df_train.isna().sum())

datetime               0
year                   0
day                    0
hour                   0
minute                 0
mag_avg_nt             0
bx_gsm_nt              0
by_gsm_nt              0
bz_gsm_nt              0
flow_speed_km_s        0
proton_density_n_cc    0
day_cos                0
day_sin                0
hour_cos               0
hour_sin               0
minute_cos             0
minute_sin             0
kp_10                  0
data_split             0
dtype: int64


In [13]:
# Check for duplicate minutes within the same year/day/hour in df_train
duplicate_minutes = df_train[
    df_train.duplicated(subset=['year', 'day', 'hour', 'minute'], keep=False)
]

print(f'Found {len(duplicate_minutes):,} rows with duplicate (year, day, hour, minute) combinations')
duplicate_minutes.sort_values(['year', 'day', 'hour', 'minute']).head(20)

Found 0 rows with duplicate (year, day, hour, minute) combinations


,datetime,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc,day_cos,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10,data_split


# II\. Create lag features \- train set

In [15]:
# Create the lagged features and drop intervals missing >5 minutes, including the lookback periods

KP_FEATURE_VARS = [
    'mag_avg_nt',
    'bx_gsm_nt',
    'by_gsm_nt',
    'bz_gsm_nt',
    'flow_speed_km_s',
    'proton_density_n_cc',
]

KP_LOOKBACK_WINDOWS = [('0_1h', 1), ('1_2h', 2), ('2_3h', 3), ('3_4h', 4)]


def build_kp_interval_features(df, value_cols=KP_FEATURE_VARS, max_missing_minutes=5):
    print(f'--- Record count audit ---')
    print(f'Input rows (per-minute): {len(df):,}')

    d = df.set_index('datetime').sort_index()

    minute_counts = d.resample('1h').size()
    hourly_stats = d[value_cols].resample('1h').agg(['mean', 'min', 'max'])
    hourly_stats.columns = [f'{col}_{stat}' for col, stat in hourly_stats.columns]
    kp_hourly_mean = d['kp_10'].resample('1h').mean()

    idx = hourly_stats.index
    print(f'Resampled hourly buckets ({idx.min()} to {idx.max()}): {len(idx):,}')

    missing = 60 - minute_counts

    # shifts past the edge of the resampled range have no underlying data, so treat them as fully missing
    window_missing = {label: missing.shift(k).fillna(60) for label, k in KP_LOOKBACK_WINDOWS}
    block_missing = missing + missing.shift(-1).fillna(60) + missing.shift(-2).fillna(60)

    interval_index = idx[idx.hour % 3 == 0]
    print(f'Candidate 3-hour intervals (hour % 3 == 0): {len(interval_index):,}')

    features = pd.DataFrame(index=interval_index)
    features.index.name = 'datetime'
    # Kp is only reported once per 3-hour block (the same value repeats across all 3 hours),
    # so the interval's own hourly value is the block value - no averaging across hours needed.
    features['kp_10'] = kp_hourly_mean.loc[interval_index]
    features['kp_index'] = (features['kp_10'] / 10.0).round(2)
    features['year'] = interval_index.year
    features['day'] = interval_index.dayofyear
    features['hour'] = interval_index.hour
    features['minute'] = 0
    features['day_cos'] = np.cos(2 * np.pi * features['day'] / 365)
    features['day_sin'] = np.sin(2 * np.pi * features['day'] / 365)
    features['hour_cos'] = np.cos(2 * np.pi * features['hour'] / 24)
    features['hour_sin'] = np.sin(2 * np.pi * features['hour'] / 24)
    features['minute_cos'] = 1.0
    features['minute_sin'] = 0.0
    features['data_split'] = df['data_split'].iloc[0]

    for label, shift_by in KP_LOOKBACK_WINDOWS:
        for var in value_cols:
            features[f'{var}_avg_{label}'] = hourly_stats[f'{var}_mean'].shift(shift_by).loc[interval_index]
            features[f'{var}_min_{label}'] = hourly_stats[f'{var}_min'].shift(shift_by).loc[interval_index]
            features[f'{var}_max_{label}'] = hourly_stats[f'{var}_max'].shift(shift_by).loc[interval_index]

    print(f'Assembled feature rows (before drop filtering): {len(features):,}, columns: {features.shape[1]:,}')

    fail_block = block_missing.loc[interval_index] > max_missing_minutes
    fail_windows = {
        label: window_missing[label].loc[interval_index] > max_missing_minutes
        for label, _ in KP_LOOKBACK_WINDOWS
    }
    drop_mask = fail_block.copy()
    for label in fail_windows:
        drop_mask |= fail_windows[label]

    reasons = pd.DataFrame({'block_3h': fail_block}, index=interval_index)
    for label in fail_windows:
        reasons[label] = fail_windows[label]
    dropped_reasons = reasons.loc[drop_mask]

    print(f'Evaluated {len(interval_index):,} candidate 3-hour Kp intervals '
          f'({interval_index.min()} to {interval_index.max()})')
    print(f'Dropping {drop_mask.sum():,} intervals ({drop_mask.mean() * 100:.2f}%) '
          f'for more than {max_missing_minutes} missing minutes in the 3-hour interval '
          f'or a lookback window:')
    print(f'  - 3-hour interval itself: {fail_block.sum():,}')
    for label, _ in KP_LOOKBACK_WINDOWS:
        print(f'  - {label} lookback window: {fail_windows[label].sum():,}')
    print(f'Keeping {(~drop_mask).sum():,} intervals')

    if drop_mask.any():
        print('\nSample of dropped intervals and failing checks:')
        print(dropped_reasons.head(10))
        if len(dropped_reasons) > 10:
            print('...')
            print(dropped_reasons.tail(5))

    kept = features.loc[~drop_mask.values].copy()

    print(f'\n--- Record count audit summary ---')
    print(f'{"Input per-minute rows:":<35}{len(df):>12,}')
    print(f'{"Resampled hourly buckets:":<35}{len(idx):>12,}')
    print(f'{"Candidate 3-hour intervals:":<35}{len(interval_index):>12,}')
    print(f'{"Dropped intervals:":<35}{drop_mask.sum():>12,}')
    print(f'{"Final kept intervals:":<35}{len(kept):>12,}')
    print(f'{"Final output shape:":<35}{str(kept.shape):>12}')
    assert len(kept) == len(interval_index) - drop_mask.sum(), 'kept count does not reconcile with candidates - drops'

    return kept, dropped_reasons

In [17]:
# Train set - Build time lag features and drop incomplete intervals

kp_features_train, kp_dropped_train = build_kp_interval_features(df_train)
kp_features_train.head()

--- Record count audit ---
Input rows (per-minute): 12,135,459
Resampled hourly buckets (2000-01-01 00:00:00 to 2023-06-30 23:00:00): 205,968
Candidate 3-hour intervals (hour % 3 == 0): 68,656
Assembled feature rows (before drop filtering): 68,656, columns: 85
Evaluated 68,656 candidate 3-hour Kp intervals (2000-01-01 00:00:00 to 2023-06-30 21:00:00)
Dropping 3,707 intervals (5.40%) for more than 5 missing minutes in the 3-hour interval or a lookback window:
  - 3-hour interval itself: 2,320
  - 0_1h lookback window: 1,494
  - 1_2h lookback window: 1,628
  - 2_3h lookback window: 1,480
  - 3_4h lookback window: 1,495
Keeping 64,949 intervals

Sample of dropped intervals and failing checks:
                     block_3h   0_1h   1_2h   2_3h   3_4h
datetime                                                 
2000-01-01 00:00:00     False   True   True   True   True
2000-01-01 03:00:00     False  False  False  False   True
2000-01-17 12:00:00      True  False  False  False  False
2000-01-17 

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,1.000000e+00,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.57
2000-01-01 09:00:00,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.35
2000-01-01 12:00:00,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,1.224647e-16,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.39
2000-01-01 15:00:00,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.40
2000-01-01 18:00:00,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,-1.000000e+00,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.25


In [19]:
kp_features_train.shape

(64949, 85)

# III\. Load test set

In [21]:
df_test = pd.read_parquet(PROCESSED_DIR / 'omni_test_cleaned.parquet')
df_test.head(5)

,datetime,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc,day_cos,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10,data_split
0,2023-06-30 23:59:00,2023,181,23,59,6.94,-3.93,4.75,2.79,515.00,1.530,-0.999667,0.025818,0.965926,-0.258819,0.994522,-0.104528,0,test
1,2023-07-01 00:00:00,2023,182,0,0,7.04,-3.64,4.16,4.27,515.00,1.530,-0.999963,0.008607,1.000000,0.000000,1.000000,0.000000,20,test
2,2023-07-01 00:01:00,2023,182,0,1,7.03,-3.76,4.18,4.17,503.25,2.115,-0.999963,0.008607,1.000000,0.000000,0.994522,0.104528,20,test
3,2023-07-01 00:02:00,2023,182,0,2,7.06,-3.88,4.25,4.02,491.50,2.700,-0.999963,0.008607,1.000000,0.000000,0.978148,0.207912,20,test
4,2023-07-01 00:03:00,2023,182,0,3,6.38,-2.79,4.11,3.72,479.75,3.285,-0.999963,0.008607,1.000000,0.000000,0.951057,0.309017,20,test


In [23]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1519035 entries, 0 to 1578240
Data columns (total 19 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   datetime             1519035 non-null  datetime64[us]
 1   year                 1519035 non-null  int64         
 2   day                  1519035 non-null  int64         
 3   hour                 1519035 non-null  int64         
 4   minute               1519035 non-null  int64         
 5   mag_avg_nt           1519035 non-null  float64       
 6   bx_gsm_nt            1519035 non-null  float64       
 7   by_gsm_nt            1519035 non-null  float64       
 8   bz_gsm_nt            1519035 non-null  float64       
 9   flow_speed_km_s      1519035 non-null  float64       
 10  proton_density_n_cc  1519035 non-null  float64       
 11  day_cos              1519035 non-null  float64       
 12  day_sin              1519035 non-null  float64       
 13  ho

In [25]:
# Confirm no nulls survived cleaning
print(df_test.isna().sum())

datetime               0
year                   0
day                    0
hour                   0
minute                 0
mag_avg_nt             0
bx_gsm_nt              0
by_gsm_nt              0
bz_gsm_nt              0
flow_speed_km_s        0
proton_density_n_cc    0
day_cos                0
day_sin                0
hour_cos               0
hour_sin               0
minute_cos             0
minute_sin             0
kp_10                  0
data_split             0
dtype: int64


# IV\. Create lag features \- test set

In [27]:
# Test set - Build time lag features and drop incomplete intervals

kp_features_test, kp_dropped_test = build_kp_interval_features(df_test)
kp_features_test.head()

--- Record count audit ---
Input rows (per-minute): 1,519,035
Resampled hourly buckets (2023-06-30 23:00:00 to 2026-06-30 23:00:00): 26,305
Candidate 3-hour intervals (hour % 3 == 0): 8,768
Assembled feature rows (before drop filtering): 8,768, columns: 85
Evaluated 8,768 candidate 3-hour Kp intervals (2023-07-01 00:00:00 to 2026-06-30 21:00:00)
Dropping 610 intervals (6.96%) for more than 5 missing minutes in the 3-hour interval or a lookback window:
  - 3-hour interval itself: 451
  - 0_1h lookback window: 361
  - 1_2h lookback window: 380
  - 2_3h lookback window: 361
  - 3_4h lookback window: 362
Keeping 8,158 intervals

Sample of dropped intervals and failing checks:
                     block_3h   0_1h   1_2h   2_3h   3_4h
datetime                                                 
2023-07-01 00:00:00     False   True   True   True   True
2023-07-01 03:00:00     False  False  False  False   True
2023-07-01 06:00:00      True  False  False  False  False
2023-07-01 09:00:00      True

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,-1.000000,...,5.95,-1.277393,-1.78,-0.89,487.209734,481.575431,512.6,2.007824,1.45,2.13
2023-07-01 21:00:00,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,-0.707107,...,3.46,0.402750,-0.74,1.55,442.770417,434.500000,449.7,1.721333,1.53,2.00
2023-07-02 00:00:00,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,0.000000,...,-0.42,1.983583,1.08,2.86,432.884167,420.100000,446.8,2.695917,2.18,3.29
2023-07-02 03:00:00,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,0.707107,...,4.75,1.654667,0.11,2.84,427.200000,414.000000,442.4,3.526167,2.97,3.91
2023-07-02 06:00:00,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,1.000000,...,5.15,-0.022833,-2.05,0.83,432.256667,419.700000,437.0,3.129833,2.71,3.68


In [29]:
kp_features_test.shape

(8158, 85)

In [31]:
# Validate the record counts pre- and post-cleaning

# Numbers sourced from 'train_test_split_before_cleaning' notebook
raw_count = 13936380
train_count = 12358079
test_count = 1578301

# Numbers sourced from 'Data Preprocessing P3'
train_dropped_count = 222620
test_dropped_count = 68684

train_cleaned_count = len(df_train)
test_cleaned_count = len(df_test)

# Assert statements
assert raw_count == train_count + test_count
assert train_count - train_dropped_count == train_cleaned_count
assert test_count - test_dropped_count == test_cleaned_count

AssertionError: 

# V\. Final sanity checks

## 1\. Kp index

In [33]:
# Look up Kp value for 2012-03-07 06:32:00
kp_features_train[
    (kp_features_train.index >= '2012-03-07 00:00') & (kp_features_train.index < '2012-03-07 21:00')
]

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2012-03-07 00:00:00,37.0,3.7,2012,67,0,0,0.405426,0.914128,1.000000e+00,0.000000e+00,...,1.880000,-4.629056,-5.990000,-2.210000,374.407500,371.800000,381.900000,5.029917,4.3900,5.77
2012-03-07 03:00:00,47.0,4.7,2012,67,3,0,0.405426,0.914128,7.071068e-01,7.071068e-01,...,-1.870000,1.263833,-2.430000,3.590000,369.182500,359.700000,393.800000,8.650000,7.7600,9.74
2012-03-07 06:00:00,60.0,6.0,2012,67,6,0,0.405426,0.914128,6.123234e-17,1.000000e+00,...,-1.650000,-2.180833,-4.590000,4.460000,367.457500,352.200000,380.300000,8.492000,6.0600,9.82
2012-03-07 09:00:00,57.0,5.7,2012,67,9,0,0.405426,0.914128,-7.071068e-01,7.071068e-01,...,-0.080000,-15.978500,-17.590000,-11.970000,423.430000,413.300000,435.400000,13.124333,9.8200,17.73
2012-03-07 12:00:00,53.0,5.3,2012,67,12,0,0.405426,0.914128,-1.000000e+00,1.224647e-16,...,1.420000,-14.136667,-19.170000,2.830000,416.434167,404.500000,422.400000,16.481500,12.6100,22.44
2012-03-07 15:00:00,53.0,5.3,2012,67,15,0,0.405426,0.914128,-7.071068e-01,-7.071068e-01,...,13.025285,1.069634,-0.952195,3.091463,467.679839,454.119355,481.240323,10.666250,8.8225,12.51
2012-03-07 18:00:00,47.0,4.7,2012,67,18,0,0.405426,0.914128,-1.836970e-16,-1.000000e+00,...,13.640000,-11.872333,-15.980000,-0.220000,523.421389,505.100000,537.500000,13.372722,10.6800,16.08


## 2\. Flow speed

In [35]:
# Identify a sample day in kp_features_train where none of the 3-hour intervals were dropped

kept_day_counts = kp_features_train.groupby(kp_features_train.index.normalize()).size()
dropped_days = set(kp_dropped_train.index.normalize())

clean_days = kept_day_counts[
    (kept_day_counts == 8) & (~kept_day_counts.index.isin(dropped_days))
]

sample_day = clean_days.index[0]
print(f'Sample day with all 8 candidate intervals kept, none dropped: {sample_day.date()}')

full_day = kp_features_train[kp_features_train.index.normalize() == sample_day]

full_day['flow_speed_km_s_avg_0_1h'].head()

Sample day with all 8 candidate intervals kept, none dropped: 2000-01-02


datetime
2000-01-02 00:00:00    706.413333
2000-01-02 03:00:00    700.590000
2000-01-02 06:00:00    694.537500
2000-01-02 09:00:00    667.896667
2000-01-02 12:00:00    669.156667
Name: flow_speed_km_s_avg_0_1h, dtype: float64

In [37]:
# Filter df_train to 2000-01-02 flow speed, hour 2

flow_speed_df = df_train[
    (df_train['year'] == 2000) &
    (df_train['day'] == 2) &
    (df_train['hour'] == 2)
]

# flow_speed_df.head()

In [39]:
# Calculate min/max/average flow speed

# Ensure datetime is stored as a pandas datetime
flow_speed_df['datetime'] = pd.to_datetime(flow_speed_df['datetime'])

# Calculate the min for this window
min_flow_speed_0_1hr = flow_speed_df['flow_speed_km_s'].min()
print(f"Min flow speed: {min_flow_speed_0_1hr}")

# Calculate the max for this window
max_flow_speed_0_1hr = flow_speed_df['flow_speed_km_s'].max()
print(f"Max flow speed: {max_flow_speed_0_1hr}")

# Calculate the average for this window
avg_flow_speed_0_1hr = flow_speed_df['flow_speed_km_s'].mean()
print(f"Average flow speed: {avg_flow_speed_0_1hr}")

Min flow speed: 677.7
Max flow speed: 728.8
Average flow speed: 700.5899999999999
/tmp/ipykernel_868/833572003.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flow_speed_df['datetime'] = pd.to_datetime(flow_speed_df['datetime'])


In [41]:
# Create df with only 2000-01-02 at 03:00

kp_features_flow = kp_features_train[
    kp_features_train.index == pd.Timestamp('2000-01-02 03:00:00')
    ]

In [43]:
kp_features_flow['flow_speed_km_s_avg_0_1h']

datetime
2000-01-02 03:00:00    700.59
Name: flow_speed_km_s_avg_0_1h, dtype: float64

In [45]:
# Report feature lag values for the 0-1 hour window

# 02:00-02:59 (2000-01-02) is the 0_1h feature window for a 03:00 prediction time

print(f"Min flow speed: {kp_features_flow['flow_speed_km_s_min_0_1h'].iloc[0]}")
print(f"Max flow speed: {kp_features_flow['flow_speed_km_s_max_0_1h'].iloc[0]}")
print(f"Avg flow speed: {kp_features_flow['flow_speed_km_s_avg_0_1h'].iloc[0]}")

Min flow speed: 677.7
Max flow speed: 728.8
Avg flow speed: 700.59


In [47]:
# Assert statements to validate min/max df features match the manual calculations
assert kp_features_flow['flow_speed_km_s_min_0_1h'].iloc[0] == min_flow_speed_0_1hr
assert kp_features_flow['flow_speed_km_s_max_0_1h'].iloc[0] == max_flow_speed_0_1hr

In [49]:
# Assert statements to validate avg df feature matches the manual calculation
features_avg_flow_speed_0_1hr = kp_features_flow['flow_speed_km_s_avg_0_1h'].iloc[0]
features_avg_flow_speed_0_1hr = round(features_avg_flow_speed_0_1hr, 3)

minute_data_avg_flow_speed_0_1hr = round(avg_flow_speed_0_1hr, 3)

assert features_avg_flow_speed_0_1hr == minute_data_avg_flow_speed_0_1hr

# VI\. Final export to parquet

In [51]:
# Write engineered Kp interval features to parquet
train_output_path = PROCESSED_DIR / 'lr_features_0h_train.parquet'
test_output_path = PROCESSED_DIR / 'lr_features_0h_test.parquet'

kp_features_train.to_parquet(train_output_path)
kp_features_test.to_parquet(test_output_path)

print(f'Wrote {len(kp_features_train):,} rows to {train_output_path.as_posix()}')
print(f'Wrote {len(kp_features_test):,} rows to {test_output_path.as_posix()}')

Wrote 64,949 rows to data/Processed/lr_features_0h_train.parquet
Wrote 8,158 rows to data/Processed/lr_features_0h_test.parquet


In [53]:
df_features_train = pd.read_parquet(PROCESSED_DIR / 'lr_features_0h_train.parquet')
df_features_test = pd.read_parquet(PROCESSED_DIR / 'lr_features_0h_test.parquet')

In [55]:
df_features_train.head(5)

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,1.000000e+00,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.57
2000-01-01 09:00:00,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.35
2000-01-01 12:00:00,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,1.224647e-16,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.39
2000-01-01 15:00:00,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.40
2000-01-01 18:00:00,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,-1.000000e+00,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.25


In [57]:
df_features_train.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 64949 entries, 2000-01-01 06:00:00 to 2023-06-30 21:00:00
Data columns (total 85 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_10                         64949 non-null  float64
 1   kp_index                      64949 non-null  float64
 2   year                          64949 non-null  int32  
 3   day                           64949 non-null  int32  
 4   hour                          64949 non-null  int32  
 5   minute                        64949 non-null  int64  
 6   day_cos                       64949 non-null  float64
 7   day_sin                       64949 non-null  float64
 8   hour_cos                      64949 non-null  float64
 9   hour_sin                      64949 non-null  float64
 10  minute_cos                    64949 non-null  float64
 11  minute_sin                    64949 non-null  float64
 12  data_split               

In [59]:
df_features_test.head(5)

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,-1.000000,...,5.95,-1.277393,-1.78,-0.89,487.209734,481.575431,512.6,2.007824,1.45,2.13
2023-07-01 21:00:00,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,-0.707107,...,3.46,0.402750,-0.74,1.55,442.770417,434.500000,449.7,1.721333,1.53,2.00
2023-07-02 00:00:00,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,0.000000,...,-0.42,1.983583,1.08,2.86,432.884167,420.100000,446.8,2.695917,2.18,3.29
2023-07-02 03:00:00,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,0.707107,...,4.75,1.654667,0.11,2.84,427.200000,414.000000,442.4,3.526167,2.97,3.91
2023-07-02 06:00:00,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,1.000000,...,5.15,-0.022833,-2.05,0.83,432.256667,419.700000,437.0,3.129833,2.71,3.68


In [61]:
df_features_test.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8158 entries, 2023-07-01 18:00:00 to 2026-06-30 21:00:00
Data columns (total 85 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_10                         8158 non-null   float64
 1   kp_index                      8158 non-null   float64
 2   year                          8158 non-null   int32  
 3   day                           8158 non-null   int32  
 4   hour                          8158 non-null   int32  
 5   minute                        8158 non-null   int64  
 6   day_cos                       8158 non-null   float64
 7   day_sin                       8158 non-null   float64
 8   hour_cos                      8158 non-null   float64
 9   hour_sin                      8158 non-null   float64
 10  minute_cos                    8158 non-null   float64
 11  minute_sin                    8158 non-null   float64
 12  data_split                

# VII\. Create forecast targets

In [63]:
VALID_FORECAST_HORIZONS = {3, 6, 9, 12, 15, 18, 21}

def create_forecast_target(df, target_column='kp_index', forecast_horizon=3):
    if forecast_horizon not in VALID_FORECAST_HORIZONS:
        raise ValueError(
            f"forecast_horizon must be one of {sorted(VALID_FORECAST_HORIZONS)}, got {forecast_horizon}"
        )

    # Rows are already 3-hour intervals, so each 3hr step in the horizon shifts one row forward
    df = df.copy()
    row_shift = forecast_horizon // 3
    new_column = f"{target_column}_{forecast_horizon}_hr_forecast"
    df[new_column] = df[target_column].shift(-row_shift)
    return df[[new_column] + [col for col in df.columns if col != new_column]]

## 3\-hour forecast

In [65]:
# Create 3-hour forecast dataframes (train/test)
kp_features_train_3hr = create_forecast_target(df=kp_features_train)
kp_features_test_3hr = create_forecast_target(df=kp_features_test)

In [67]:
kp_features_train_3hr.info()
kp_features_train_3hr.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 64949 entries, 2000-01-01 06:00:00 to 2023-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_index_3_hr_forecast        64948 non-null  float64
 1   kp_10                         64949 non-null  float64
 2   kp_index                      64949 non-null  float64
 3   year                          64949 non-null  int32  
 4   day                           64949 non-null  int32  
 5   hour                          64949 non-null  int32  
 6   minute                        64949 non-null  int64  
 7   day_cos                       64949 non-null  float64
 8   day_sin                       64949 non-null  float64
 9   hour_cos                      64949 non-null  float64
 10  hour_sin                      64949 non-null  float64
 11  minute_cos                    64949 non-null  float64
 12  minute_sin               

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,3.3,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.57
2000-01-01 09:00:00,4.3,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.35
2000-01-01 12:00:00,3.0,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.39
2000-01-01 15:00:00,4.3,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.40
2000-01-01 18:00:00,3.7,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.25


In [69]:
# Compare to original kp_index values
kp_features_train.head()

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,1.000000e+00,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.57
2000-01-01 09:00:00,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.35
2000-01-01 12:00:00,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,1.224647e-16,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.39
2000-01-01 15:00:00,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.40
2000-01-01 18:00:00,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,-1.000000e+00,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.25


In [71]:
# Check for nulls after shift
kp_features_train_3hr[kp_features_train_3hr.isnull().any(axis=1)]

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-30 21:00:00,NaN,0.0,0.0,2023,181,21,0,-0.999667,0.025818,0.707107,...,3.11,1.705917,1.05,2.01,513.715,510.6,520.6,1.6479,1.52,1.82


In [73]:
# Drop resulting nulls for train set
print(f"Records before dropping NAs: {len(kp_features_train_3hr):,}")
kp_features_train_3hr = kp_features_train_3hr.dropna(subset=['kp_index_3_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_train_3hr):,}")

Records before dropping NAs: 64,949
Records after dropping NAs: 64,948


In [75]:
kp_features_test_3hr.info()
kp_features_test_3hr.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8158 entries, 2023-07-01 18:00:00 to 2026-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_index_3_hr_forecast        8157 non-null   float64
 1   kp_10                         8158 non-null   float64
 2   kp_index                      8158 non-null   float64
 3   year                          8158 non-null   int32  
 4   day                           8158 non-null   int32  
 5   hour                          8158 non-null   int32  
 6   minute                        8158 non-null   int64  
 7   day_cos                       8158 non-null   float64
 8   day_sin                       8158 non-null   float64
 9   hour_cos                      8158 non-null   float64
 10  hour_sin                      8158 non-null   float64
 11  minute_cos                    8158 non-null   float64
 12  minute_sin                

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,1.3,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,...,5.95,-1.277393,-1.78,-0.89,487.209734,481.575431,512.6,2.007824,1.45,2.13
2023-07-01 21:00:00,0.7,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,...,3.46,0.402750,-0.74,1.55,442.770417,434.500000,449.7,1.721333,1.53,2.00
2023-07-02 00:00:00,1.3,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,...,-0.42,1.983583,1.08,2.86,432.884167,420.100000,446.8,2.695917,2.18,3.29
2023-07-02 03:00:00,0.7,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,...,4.75,1.654667,0.11,2.84,427.200000,414.000000,442.4,3.526167,2.97,3.91
2023-07-02 06:00:00,1.3,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,...,5.15,-0.022833,-2.05,0.83,432.256667,419.700000,437.0,3.129833,2.71,3.68


In [77]:
# Check for nulls after shift
kp_features_test_3hr[kp_features_test_3hr.isnull().any(axis=1)]

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2026-06-30 21:00:00,NaN,33.0,3.3,2026,181,21,0,-0.999667,0.025818,0.707107,...,-4.83,-7.477333,-8.72,-6.15,427.5,420.9,435.0,16.659667,13.43,20.08


In [79]:
# Drop resulting nulls for test set
print(f"Records before dropping NAs: {len(kp_features_test_3hr):,}")
kp_features_test_3hr = kp_features_test_3hr.dropna(subset=['kp_index_3_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_test_3hr):,}")

Records before dropping NAs: 8,158
Records after dropping NAs: 8,157


In [81]:
# Write engineered Kp interval features to parquet
train_output_path = PROCESSED_DIR / 'lr_features_3h_train.parquet'
test_output_path = PROCESSED_DIR / 'lr_features_3h_test.parquet'

kp_features_train_3hr.to_parquet(train_output_path)
kp_features_test_3hr.to_parquet(test_output_path)

print(f'Wrote {len(kp_features_train_3hr):,} rows to {train_output_path.as_posix()}')
print(f'Wrote {len(kp_features_test_3hr):,} rows to {test_output_path.as_posix()}')

Wrote 64,948 rows to data/Processed/lr_features_3h_train.parquet
Wrote 8,157 rows to data/Processed/lr_features_3h_test.parquet


## 6\-hour forecast

In [83]:
# Create 6-hour forecast dataframes (train/test)
kp_features_train_6hr = create_forecast_target(df=kp_features_train, forecast_horizon=6)
kp_features_test_6hr = create_forecast_target(df=kp_features_test, forecast_horizon=6)

In [85]:
# Check train set
kp_features_train_6hr.info()
kp_features_train_6hr.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 64949 entries, 2000-01-01 06:00:00 to 2023-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_index_6_hr_forecast        64947 non-null  float64
 1   kp_10                         64949 non-null  float64
 2   kp_index                      64949 non-null  float64
 3   year                          64949 non-null  int32  
 4   day                           64949 non-null  int32  
 5   hour                          64949 non-null  int32  
 6   minute                        64949 non-null  int64  
 7   day_cos                       64949 non-null  float64
 8   day_sin                       64949 non-null  float64
 9   hour_cos                      64949 non-null  float64
 10  hour_sin                      64949 non-null  float64
 11  minute_cos                    64949 non-null  float64
 12  minute_sin               

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,4.3,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.57
2000-01-01 09:00:00,3.0,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.35
2000-01-01 12:00:00,4.3,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.39
2000-01-01 15:00:00,3.7,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.40
2000-01-01 18:00:00,3.0,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.25


In [87]:
# Compare to original kp_index values
kp_features_train.head()

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,1.000000e+00,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.57
2000-01-01 09:00:00,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.35
2000-01-01 12:00:00,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,1.224647e-16,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.39
2000-01-01 15:00:00,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.40
2000-01-01 18:00:00,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,-1.000000e+00,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.25


In [89]:
# Check for nulls after shift
kp_features_train_6hr[kp_features_train_6hr.isnull().any(axis=1)]

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-30 18:00:00,NaN,3.0,0.3,2023,181,18,0,-0.999667,0.025818,-1.836970e-16,...,5.00,-1.245833,-2.60,0.27,558.235833,521.2,603.9,2.7990,1.71,3.79
2023-06-30 21:00:00,NaN,0.0,0.0,2023,181,21,0,-0.999667,0.025818,7.071068e-01,...,3.11,1.705917,1.05,2.01,513.715000,510.6,520.6,1.6479,1.52,1.82


In [91]:
# Drop resulting nulls for train set
print(f"Records before dropping NAs: {len(kp_features_train_6hr):,}")
kp_features_train_6hr = kp_features_train_6hr.dropna(subset=['kp_index_6_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_train_6hr):,}")

Records before dropping NAs: 64,949
Records after dropping NAs: 64,947


In [93]:
# Check test set
kp_features_test_6hr.info()
kp_features_test_6hr.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8158 entries, 2023-07-01 18:00:00 to 2026-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_index_6_hr_forecast        8156 non-null   float64
 1   kp_10                         8158 non-null   float64
 2   kp_index                      8158 non-null   float64
 3   year                          8158 non-null   int32  
 4   day                           8158 non-null   int32  
 5   hour                          8158 non-null   int32  
 6   minute                        8158 non-null   int64  
 7   day_cos                       8158 non-null   float64
 8   day_sin                       8158 non-null   float64
 9   hour_cos                      8158 non-null   float64
 10  hour_sin                      8158 non-null   float64
 11  minute_cos                    8158 non-null   float64
 12  minute_sin                

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,0.7,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,...,5.95,-1.277393,-1.78,-0.89,487.209734,481.575431,512.6,2.007824,1.45,2.13
2023-07-01 21:00:00,1.3,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,...,3.46,0.402750,-0.74,1.55,442.770417,434.500000,449.7,1.721333,1.53,2.00
2023-07-02 00:00:00,0.7,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,...,-0.42,1.983583,1.08,2.86,432.884167,420.100000,446.8,2.695917,2.18,3.29
2023-07-02 03:00:00,1.3,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,...,4.75,1.654667,0.11,2.84,427.200000,414.000000,442.4,3.526167,2.97,3.91
2023-07-02 06:00:00,1.7,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,...,5.15,-0.022833,-2.05,0.83,432.256667,419.700000,437.0,3.129833,2.71,3.68


In [95]:
# Check for nulls after shift
kp_features_test_6hr[kp_features_test_6hr.isnull().any(axis=1)]

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2026-06-30 18:00:00,NaN,47.0,4.7,2026,181,18,0,-0.999667,0.025818,-1.836970e-16,...,-7.14,-0.802500,-6.45,7.81,434.704167,427.2,443.5,11.910250,9.45,13.52
2026-06-30 21:00:00,NaN,33.0,3.3,2026,181,21,0,-0.999667,0.025818,7.071068e-01,...,-4.83,-7.477333,-8.72,-6.15,427.500000,420.9,435.0,16.659667,13.43,20.08


In [97]:
# Drop resulting nulls for test set
print(f"Records before dropping NAs: {len(kp_features_test_6hr):,}")
kp_features_test_6hr = kp_features_test_6hr.dropna(subset=['kp_index_6_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_test_6hr):,}")

Records before dropping NAs: 8,158
Records after dropping NAs: 8,156


In [99]:
# Write engineered Kp interval features to parquet
train_output_path = PROCESSED_DIR / 'lr_features_6h_train.parquet'
test_output_path = PROCESSED_DIR / 'lr_features_6h_test.parquet'

kp_features_train_6hr.to_parquet(train_output_path)
kp_features_test_6hr.to_parquet(test_output_path)

print(f'Wrote {len(kp_features_train_6hr):,} rows to {train_output_path.as_posix()}')
print(f'Wrote {len(kp_features_test_6hr):,} rows to {test_output_path.as_posix()}')

Wrote 64,947 rows to data/Processed/lr_features_6h_train.parquet
Wrote 8,156 rows to data/Processed/lr_features_6h_test.parquet


## 9\-hour forecast

In [101]:
# Create 9-hour forecast dataframes (train/test)
kp_features_train_9hr = create_forecast_target(df=kp_features_train, forecast_horizon=9)
kp_features_test_9hr = create_forecast_target(df=kp_features_test, forecast_horizon=9)

In [103]:
# Check train set
kp_features_train_9hr.info()
kp_features_train_9hr.head(10)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 64949 entries, 2000-01-01 06:00:00 to 2023-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_index_9_hr_forecast        64946 non-null  float64
 1   kp_10                         64949 non-null  float64
 2   kp_index                      64949 non-null  float64
 3   year                          64949 non-null  int32  
 4   day                           64949 non-null  int32  
 5   hour                          64949 non-null  int32  
 6   minute                        64949 non-null  int64  
 7   day_cos                       64949 non-null  float64
 8   day_sin                       64949 non-null  float64
 9   hour_cos                      64949 non-null  float64
 10  hour_sin                      64949 non-null  float64
 11  minute_cos                    64949 non-null  float64
 12  minute_sin               

,kp_index_9_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,3.0,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.570
2000-01-01 09:00:00,4.3,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.350
2000-01-01 12:00:00,3.7,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.390
2000-01-01 15:00:00,3.0,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.400
2000-01-01 18:00:00,3.3,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.250
2000-01-01 21:00:00,3.3,37.0,3.7,2000,1,21,0,0.999852,0.017213,7.071068e-01,...,4.34,0.694250,-2.02,4.50,684.890833,612.6,732.2,3.175417,1.87,5.150
2000-01-02 00:00:00,3.3,30.0,3.0,2000,2,0,0,0.999407,0.034422,1.000000e+00,...,6.31,0.203833,-3.04,2.64,719.367500,697.5,751.5,1.714667,1.49,2.385
2000-01-02 03:00:00,2.7,33.0,3.3,2000,2,3,0,0.999407,0.034422,7.071068e-01,...,5.50,0.573750,-6.12,5.45,706.413333,665.0,771.8,2.825167,1.97,4.090
2000-01-02 06:00:00,3.3,33.0,3.3,2000,2,6,0,0.999407,0.034422,6.123234e-17,...,4.65,2.583583,-4.02,5.15,700.590000,677.7,728.8,2.258167,1.91,2.560


In [105]:
# Compare to original kp_index values
kp_features_train.head(10)

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,1.000000e+00,...,7.51,-2.650083,-7.15,4.43,712.120000,676.2,772.9,2.185500,1.66,2.570
2000-01-01 09:00:00,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,7.071068e-01,...,5.37,-0.691833,-4.90,4.35,718.101667,689.3,772.1,1.952167,1.75,2.350
2000-01-01 12:00:00,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,1.224647e-16,...,6.39,-0.399500,-6.13,6.58,749.054167,715.5,775.6,1.854750,1.45,2.390
2000-01-01 15:00:00,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,726.491667,703.3,764.5,2.109917,1.77,2.400
2000-01-01 18:00:00,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,-1.000000e+00,...,5.81,-0.374333,-2.88,4.80,708.437500,682.9,756.3,3.066083,1.69,4.250
2000-01-01 21:00:00,37.0,3.7,2000,1,21,0,0.999852,0.017213,7.071068e-01,-7.071068e-01,...,4.34,0.694250,-2.02,4.50,684.890833,612.6,732.2,3.175417,1.87,5.150
2000-01-02 00:00:00,30.0,3.0,2000,2,0,0,0.999407,0.034422,1.000000e+00,0.000000e+00,...,6.31,0.203833,-3.04,2.64,719.367500,697.5,751.5,1.714667,1.49,2.385
2000-01-02 03:00:00,33.0,3.3,2000,2,3,0,0.999407,0.034422,7.071068e-01,7.071068e-01,...,5.50,0.573750,-6.12,5.45,706.413333,665.0,771.8,2.825167,1.97,4.090
2000-01-02 06:00:00,33.0,3.3,2000,2,6,0,0.999407,0.034422,6.123234e-17,1.000000e+00,...,4.65,2.583583,-4.02,5.15,700.590000,677.7,728.8,2.258167,1.91,2.560


In [107]:
# Check for nulls after shift
kp_features_train_9hr[kp_features_train_9hr.isnull().any(axis=1)]

,kp_index_9_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-30 15:00:00,NaN,17.0,1.7,2023,181,15,0,-0.999667,0.025818,-7.071068e-01,...,2.44,-2.945750,-4.14,-1.86,493.820000,474.1,513.7,2.060333,1.57,3.12
2023-06-30 18:00:00,NaN,3.0,0.3,2023,181,18,0,-0.999667,0.025818,-1.836970e-16,...,5.00,-1.245833,-2.60,0.27,558.235833,521.2,603.9,2.799000,1.71,3.79
2023-06-30 21:00:00,NaN,0.0,0.0,2023,181,21,0,-0.999667,0.025818,7.071068e-01,...,3.11,1.705917,1.05,2.01,513.715000,510.6,520.6,1.647900,1.52,1.82


In [109]:
# Drop resulting nulls for train set
print(f"Records before dropping NAs: {len(kp_features_train_9hr):,}")
kp_features_train_9hr = kp_features_train_9hr.dropna(subset=['kp_index_9_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_train_9hr):,}")

Records before dropping NAs: 64,949
Records after dropping NAs: 64,946


In [111]:
# Check test set
kp_features_test_9hr.info()
kp_features_test_9hr.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8158 entries, 2023-07-01 18:00:00 to 2026-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kp_index_9_hr_forecast        8155 non-null   float64
 1   kp_10                         8158 non-null   float64
 2   kp_index                      8158 non-null   float64
 3   year                          8158 non-null   int32  
 4   day                           8158 non-null   int32  
 5   hour                          8158 non-null   int32  
 6   minute                        8158 non-null   int64  
 7   day_cos                       8158 non-null   float64
 8   day_sin                       8158 non-null   float64
 9   hour_cos                      8158 non-null   float64
 10  hour_sin                      8158 non-null   float64
 11  minute_cos                    8158 non-null   float64
 12  minute_sin                

,kp_index_9_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,1.3,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,...,5.95,-1.277393,-1.78,-0.89,487.209734,481.575431,512.6,2.007824,1.45,2.13
2023-07-01 21:00:00,0.7,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,...,3.46,0.402750,-0.74,1.55,442.770417,434.500000,449.7,1.721333,1.53,2.00
2023-07-02 00:00:00,1.3,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,...,-0.42,1.983583,1.08,2.86,432.884167,420.100000,446.8,2.695917,2.18,3.29
2023-07-02 03:00:00,1.7,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,...,4.75,1.654667,0.11,2.84,427.200000,414.000000,442.4,3.526167,2.97,3.91
2023-07-02 06:00:00,1.0,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,...,5.15,-0.022833,-2.05,0.83,432.256667,419.700000,437.0,3.129833,2.71,3.68


In [113]:
# Check for nulls after shift
kp_features_test_9hr[kp_features_test_9hr.isnull().any(axis=1)]

,kp_index_9_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_avg_3_4h,flow_speed_km_s_min_3_4h,flow_speed_km_s_max_3_4h,proton_density_n_cc_avg_3_4h,proton_density_n_cc_min_3_4h,proton_density_n_cc_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2026-06-30 15:00:00,NaN,47.0,4.7,2026,181,15,0,-0.999667,0.025818,-7.071068e-01,...,-1.67,-1.671814,-5.39,0.07,416.823636,409.0,421.3,11.329655,10.586795,12.66
2026-06-30 18:00:00,NaN,47.0,4.7,2026,181,18,0,-0.999667,0.025818,-1.836970e-16,...,-7.14,-0.802500,-6.45,7.81,434.704167,427.2,443.5,11.910250,9.450000,13.52
2026-06-30 21:00:00,NaN,33.0,3.3,2026,181,21,0,-0.999667,0.025818,7.071068e-01,...,-4.83,-7.477333,-8.72,-6.15,427.500000,420.9,435.0,16.659667,13.430000,20.08


In [115]:
# Drop resulting nulls for test set
print(f"Records before dropping NAs: {len(kp_features_test_9hr):,}")
kp_features_test_9hr = kp_features_test_9hr.dropna(subset=['kp_index_9_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_test_9hr):,}")

Records before dropping NAs: 8,158
Records after dropping NAs: 8,155


In [117]:
# Write engineered Kp interval features to parquet
train_output_path = PROCESSED_DIR / 'lr_features_9h_train.parquet'
test_output_path = PROCESSED_DIR / 'lr_features_9h_test.parquet'

kp_features_train_9hr.to_parquet(train_output_path)
kp_features_test_9hr.to_parquet(test_output_path)

print(f'Wrote {len(kp_features_train_9hr):,} rows to {train_output_path.as_posix()}')
print(f'Wrote {len(kp_features_test_9hr):,} rows to {test_output_path.as_posix()}')

Wrote 64,946 rows to data/Processed/lr_features_9h_train.parquet
Wrote 8,155 rows to data/Processed/lr_features_9h_test.parquet


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>